# Mixed-Type Mixture Model: Gaussian + Categorical

In the [previous notebook](gower_kmedoids.ipynb), we used **K-Medoids with Gower distance** to cluster mixed-type data: geographic coordinates (continuous) combined with climate and terrain categories (discrete). That approach worked, but it has limitations:

- **Hard assignments only** — each point belongs to exactly one cluster, with no notion of uncertainty
- **No generative model** — Gower distance is a useful heuristic, but it doesn't describe *how* the data was produced
- **No learned category distributions** — we can't ask "what's the probability of 'tropical' in cluster 3?"

## The Mixed-Type Mixture Approach

Instead of a distance-based method, we build a **probabilistic generative model** where each mixture component has:
- A **2D Gaussian** for the continuous features (latitude, longitude in radians)
- A **categorical (multinomial) distribution** for each discrete feature (climate type, terrain type)

The key insight is that the joint likelihood **factorizes** under conditional independence:

$$p(x_{\text{cont}}, x_{\text{cat}} \mid \theta_k) = p(x_{\text{cont}} \mid \mu_k, \Sigma_k) \times p(x_{\text{cat}} \mid \phi_k)$$

This is sometimes called a **latent class model** or a **mixture of Gaussians and multinomials**. It gives us everything K-Medoids couldn't: soft assignments, uncertainty quantification, and learned category probabilities per cluster.

**Analogy:** Think of each cluster as a "region profile" — it describes both *where* a station is likely to be (Gaussian) and *what kind* of station it is likely to be (categorical probabilities). The EM algorithm figures out these profiles from unlabeled data.

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

np.random.seed(42)

# Color palette (consistent across series)
COLORS = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

# Category definitions
CLIMATES = ['tropical', 'arid', 'temperate', 'polar']
TERRAINS = ['mountain', 'plains', 'forest', 'coastal']
N_CLIMATES = len(CLIMATES)
N_TERRAINS = len(TERRAINS)

# Marker styles for climate types
CLIMATE_MARKERS = ['o', 's', '^', 'D']  # circle, square, triangle, diamond

print("Setup complete!")
print(f"Climate categories: {CLIMATES}")
print(f"Terrain categories: {TERRAINS}")

## 2. Generate Mixed-Type Data

We create 5 clusters of weather stations, each with:
- **Geographic coordinates** (lat, lon) in radians — continuous features
- **Climate type** — one of 4 categories, drawn from a cluster-specific distribution
- **Terrain type** — one of 4 categories, drawn from a cluster-specific distribution

The crucial design choice: **clusters 1 and 2 are geographically close but categorically very different** (one is tropical/coastal, the other is polar/mountain). This is exactly the scenario where a mixed-type model shines — it can separate them using categorical information even when the geographic distributions overlap.

In [ ]:
def generate_mixed_data(seed=42):
    """
    Generate mixed-type data: continuous (lat, lon) + categorical (climate, terrain).
    Returns continuous features, categorical codes, true labels, and true parameters.
    """
    rng = np.random.default_rng(seed)
    
    # Cluster configurations
    # Each cluster: geographic center, spread, dominant climate probs, dominant terrain probs, n_points
    clusters = [
        {  # Cluster 0: Tropical coastal region (equatorial)
            'center': np.array([0.1, -1.2]),
            'cov': np.array([[0.04, 0.005], [0.005, 0.05]]),
            'climate_probs': [0.75, 0.10, 0.10, 0.05],  # mostly tropical
            'terrain_probs': [0.10, 0.10, 0.15, 0.65],   # mostly coastal
            'n': 60
        },
        {  # Cluster 1: Polar mountain region — CLOSE to cluster 2 geographically!
            'center': np.array([0.8, 0.5]),
            'cov': np.array([[0.03, -0.005], [-0.005, 0.04]]),
            'climate_probs': [0.05, 0.05, 0.15, 0.75],  # mostly polar
            'terrain_probs': [0.70, 0.10, 0.10, 0.10],   # mostly mountain
            'n': 55
        },
        {  # Cluster 2: Tropical forest region — CLOSE to cluster 1 geographically!
            'center': np.array([0.7, 0.6]),
            'cov': np.array([[0.03, 0.008], [0.008, 0.035]]),
            'climate_probs': [0.70, 0.10, 0.15, 0.05],  # mostly tropical
            'terrain_probs': [0.05, 0.10, 0.70, 0.15],   # mostly forest
            'n': 65
        },
        {  # Cluster 3: Arid plains region
            'center': np.array([-0.5, 1.5]),
            'cov': np.array([[0.06, 0.01], [0.01, 0.04]]),
            'climate_probs': [0.10, 0.70, 0.15, 0.05],  # mostly arid
            'terrain_probs': [0.05, 0.70, 0.15, 0.10],   # mostly plains
            'n': 60
        },
        {  # Cluster 4: Temperate mixed region
            'center': np.array([-0.3, -0.4]),
            'cov': np.array([[0.05, -0.01], [-0.01, 0.06]]),
            'climate_probs': [0.10, 0.10, 0.65, 0.15],  # mostly temperate
            'terrain_probs': [0.20, 0.20, 0.35, 0.25],   # mixed terrain
            'n': 60
        }
    ]
    
    all_continuous = []
    all_climate = []
    all_terrain = []
    all_labels = []
    
    true_params = []
    
    for i, cfg in enumerate(clusters):
        n = cfg['n']
        # Generate continuous features (lat, lon in radians)
        pts = rng.multivariate_normal(cfg['center'], cfg['cov'], size=n)
        all_continuous.append(pts)
        
        # Generate categorical features from multinomial
        climate_codes = rng.choice(N_CLIMATES, size=n, p=cfg['climate_probs'])
        terrain_codes = rng.choice(N_TERRAINS, size=n, p=cfg['terrain_probs'])
        all_climate.append(climate_codes)
        all_terrain.append(terrain_codes)
        
        all_labels.append(np.full(n, i))
        true_params.append(cfg)
    
    X_cont = np.vstack(all_continuous)
    climate = np.concatenate(all_climate)
    terrain = np.concatenate(all_terrain)
    labels = np.concatenate(all_labels).astype(int)
    
    # Shuffle
    perm = rng.permutation(len(X_cont))
    X_cont = X_cont[perm]
    climate = climate[perm]
    terrain = terrain[perm]
    labels = labels[perm]
    
    return X_cont, climate, terrain, labels, true_params


X_cont, climate, terrain, true_labels, true_params = generate_mixed_data()
N = len(X_cont)
K = 5

print(f"Generated {N} points in {K} clusters")
print(f"Cluster sizes: {[np.sum(true_labels == i) for i in range(K)]}")
print(f"Lat range: [{X_cont[:,0].min():.3f}, {X_cont[:,0].max():.3f}] rad")
print(f"Lon range: [{X_cont[:,1].min():.3f}, {X_cont[:,1].max():.3f}] rad")
print(f"\nClimate distribution: {dict(zip(CLIMATES, [np.sum(climate==i) for i in range(N_CLIMATES)]))}")
print(f"Terrain distribution: {dict(zip(TERRAINS, [np.sum(terrain==i) for i in range(N_TERRAINS)]))}")

# Note: Clusters 1 and 2 are geographically overlapping!
print(f"\n--- Key overlap ---")
print(f"Cluster 1 center: {true_params[1]['center']} (polar/mountain)")
print(f"Cluster 2 center: {true_params[2]['center']} (tropical/forest)")
print(f"Distance between centers: {np.linalg.norm(true_params[1]['center'] - true_params[2]['center']):.3f} rad")

In [ ]:
# Visualize the true data
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: colored by true cluster
ax = axes[0]
for k in range(K):
    mask = true_labels == k
    ax.scatter(X_cont[mask, 1], X_cont[mask, 0], c=COLORS[k], 
               alpha=0.6, s=40, label=f'Cluster {k}')
    ax.plot(true_params[k]['center'][1], true_params[k]['center'][0], 
            'k*', markersize=15, markeredgewidth=1.5)
ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title('True Clusters (Geographic)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: colored by climate, shaped by terrain
ax = axes[1]
climate_colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']
for c_idx in range(N_CLIMATES):
    mask = climate == c_idx
    ax.scatter(X_cont[mask, 1], X_cont[mask, 0], c=climate_colors[c_idx],
               marker=CLIMATE_MARKERS[c_idx], alpha=0.6, s=40,
               label=CLIMATES[c_idx])
ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title('Data by Climate Type', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Mixed-Type Data: Geography + Categories', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nNotice: the overlapping region (lon~0.5, lat~0.8) contains BOTH polar/mountain")
print("and tropical/forest stations. Geography alone can't separate them!")

## 3. The Mixed-Type Likelihood

For component $k$, the likelihood of observation $n$ with continuous features $x_n$ and categorical features $c_n = (c_{n1}, c_{n2}, \ldots)$:

$$p(x_n, c_n \mid \theta_k) = \mathcal{N}(x_n \mid \mu_k, \Sigma_k) \times \prod_{f} \phi_{k,f,c_{nf}}$$

where:
- $\mathcal{N}(x_n \mid \mu_k, \Sigma_k)$ is the **Gaussian density** for the continuous features (lat, lon)
- $\phi_{k,f,v}$ is the **probability** that categorical feature $f$ takes value $v$ in component $k$
- $\phi_{k,f,v} \geq 0$ and $\sum_v \phi_{k,f,v} = 1$ for each $(k, f)$

### Conditional Independence Assumption

This model assumes that, **given the component**, the continuous and categorical features are independent. This is the same "naive Bayes" style assumption — it's not always true, but it's remarkably effective and keeps the model tractable.

In log space:

$$\log p(x_n, c_n \mid \theta_k) = \log \mathcal{N}(x_n \mid \mu_k, \Sigma_k) + \sum_f \log \phi_{k,f,c_{nf}}$$

The Gaussian log-density (using Cholesky decomposition of $\Sigma_k = L_k L_k^T$):

$$\log \mathcal{N}(x \mid \mu, \Sigma) = -\frac{d}{2}\log(2\pi) - \sum_i \log L_{ii} - \frac{1}{2}\|L^{-1}(x - \mu)\|^2$$

In [ ]:
def log_gaussian_pdf(X, mu, cov, reg=1e-6):
    """
    Compute log N(x | mu, cov) for each row of X using Cholesky decomposition.
    
    Parameters:
        X: (N, D) data points
        mu: (D,) mean
        cov: (D, D) covariance matrix
        reg: regularization added to diagonal
    
    Returns:
        log_probs: (N,) log-densities
    """
    D = X.shape[1]
    # Regularize
    cov_reg = cov + reg * np.eye(D)
    
    # Cholesky: cov = L @ L.T
    L = np.linalg.cholesky(cov_reg)
    
    # Solve L @ z = (X - mu).T => z = L^{-1} (X - mu).T
    diff = X - mu  # (N, D)
    z = np.linalg.solve(L, diff.T)  # (D, N)
    
    # Mahalanobis: ||z||^2 for each point
    mahal = np.sum(z**2, axis=0)  # (N,)
    
    # Log determinant: 2 * sum(log(diag(L)))
    log_det = 2.0 * np.sum(np.log(np.diag(L)))
    
    # Log probability
    log_probs = -0.5 * (D * np.log(2 * np.pi) + log_det + mahal)
    
    return log_probs


# Quick test
test_X = np.array([[0.0, 0.0], [1.0, 1.0]])
test_mu = np.array([0.0, 0.0])
test_cov = np.eye(2)
print("Log-Gaussian PDF test:")
print(f"  log N([0,0] | [0,0], I) = {log_gaussian_pdf(test_X[:1], test_mu, test_cov)[0]:.4f}")
print(f"  Expected: {-0.5 * 2 * np.log(2*np.pi):.4f}")
print("  Match!") if np.isclose(log_gaussian_pdf(test_X[:1], test_mu, test_cov)[0], 
                                 -0.5 * 2 * np.log(2*np.pi)) else print("  Mismatch!")

## 4. EM Algorithm for Mixed-Type Mixture Model

### E-Step: Compute Responsibilities

The responsibility $r_{nk}$ is the posterior probability that point $n$ belongs to component $k$:

$$r_{nk} = \frac{\pi_k \, \mathcal{N}(x_n \mid \mu_k, \Sigma_k) \prod_f \phi_{k,f,c_{nf}}}{\sum_j \pi_j \, \mathcal{N}(x_n \mid \mu_j, \Sigma_j) \prod_f \phi_{j,f,c_{nf}}}$$

In log space (for numerical stability):

$$\log \tilde{r}_{nk} = \log \pi_k + \log \mathcal{N}(x_n \mid \mu_k, \Sigma_k) + \sum_f \log \phi_{k,f,c_{nf}}$$

Then normalize using log-sum-exp: $r_{nk} = \exp(\log \tilde{r}_{nk} - \text{logsumexp}_k(\log \tilde{r}_{nk}))$

### M-Step: Update Parameters

**Effective counts:** $N_k = \sum_n r_{nk}$

**Mixing weights:** $\pi_k = N_k / N$

**Gaussian parameters:**
$$\mu_k = \frac{\sum_n r_{nk} \, x_n}{N_k}, \qquad \Sigma_k = \frac{\sum_n r_{nk} (x_n - \mu_k)(x_n - \mu_k)^T}{N_k}$$

**Categorical parameters** (with Laplace smoothing $\alpha = 0.01$):
$$\phi_{k,f,v} = \frac{\sum_{n: c_{nf}=v} r_{nk} + \alpha}{N_k + \alpha \cdot V_f}$$

where $V_f$ is the number of categories for feature $f$. The smoothing prevents zero probabilities, which would cause log-likelihood to be $-\infty$.

### Log-Likelihood

$$\mathcal{L} = \sum_n \log \sum_k \pi_k \, p(x_n, c_n \mid \theta_k)$$

Computed via log-sum-exp over the unnormalized log-responsibilities.

In [ ]:
def mixed_em(X_cont, cat_features, n_categories, K=5, max_iter=100, tol=1e-6, 
             alpha=0.01, reg=1e-6, seed=123):
    """
    EM algorithm for a mixture of Gaussians (continuous) + Multinomials (categorical).
    
    Parameters:
        X_cont: (N, D) continuous features
        cat_features: list of F arrays, each (N,) with integer codes
        n_categories: list of F ints, number of categories per feature
        K: number of mixture components
        max_iter: maximum EM iterations
        tol: convergence tolerance on log-likelihood
        alpha: Laplace smoothing for categorical probabilities
        reg: covariance regularization
        seed: random seed for initialization
    
    Returns:
        Dictionary with learned parameters and diagnostics.
    """
    rng = np.random.default_rng(seed)
    N, D = X_cont.shape
    F = len(cat_features)  # number of categorical features
    
    # --- Initialization ---
    # Random initialization: pick K random data points as initial means
    init_idx = rng.choice(N, size=K, replace=False)
    
    # Gaussian params
    means = X_cont[init_idx].copy()  # (K, D)
    covs = np.array([np.cov(X_cont.T) * 0.5 for _ in range(K)])  # (K, D, D)
    
    # Categorical params: initialize from data + noise
    cat_probs = []  # list of F arrays, each (K, V_f)
    for f in range(F):
        V_f = n_categories[f]
        # Start with uniform + small random perturbation
        phi_f = np.ones((K, V_f)) / V_f + rng.uniform(0, 0.1, size=(K, V_f))
        phi_f /= phi_f.sum(axis=1, keepdims=True)  # normalize rows
        cat_probs.append(phi_f)
    
    # Mixing weights
    pi = np.ones(K) / K
    
    log_likelihoods = []
    
    print(f"Starting EM with K={K}, N={N}, D={D}, F={F} categorical features")
    print(f"Categories per feature: {n_categories}")
    print(f"Laplace smoothing alpha={alpha}, covariance reg={reg}")
    print("-" * 60)
    
    for iteration in range(max_iter):
        # ===================== E-STEP =====================
        # Compute log responsibilities: log r_nk (unnormalized)
        log_resp = np.zeros((N, K))
        
        for k in range(K):
            # Gaussian log-likelihood
            log_gauss = log_gaussian_pdf(X_cont, means[k], covs[k], reg=reg)  # (N,)
            
            # Categorical log-likelihood
            log_cat = np.zeros(N)
            for f in range(F):
                # phi_{k,f,c_{nf}} for each point
                log_cat += np.log(cat_probs[f][k, cat_features[f]] + 1e-300)
            
            log_resp[:, k] = np.log(pi[k] + 1e-300) + log_gauss + log_cat
        
        # Log-sum-exp for normalization and log-likelihood
        log_resp_max = np.max(log_resp, axis=1, keepdims=True)  # (N, 1)
        log_sum = log_resp_max.squeeze() + np.log(
            np.sum(np.exp(log_resp - log_resp_max), axis=1)
        )  # (N,)
        
        # Log-likelihood
        ll = np.sum(log_sum)
        log_likelihoods.append(ll)
        
        # Normalize responsibilities
        resp = np.exp(log_resp - log_sum[:, np.newaxis])  # (N, K)
        
        # Check convergence
        if iteration > 0:
            delta = ll - log_likelihoods[-2]
            if iteration % 10 == 0 or iteration < 5:
                print(f"  Iter {iteration:3d}: log-likelihood = {ll:12.2f}  (delta = {delta:+.6f})")
            if abs(delta) < tol:
                print(f"  Converged at iteration {iteration} (delta = {delta:.2e})")
                break
        else:
            print(f"  Iter {iteration:3d}: log-likelihood = {ll:12.2f}")
        
        # ===================== M-STEP =====================
        # Effective counts
        N_k = resp.sum(axis=0)  # (K,)
        
        # Mixing weights
        pi = N_k / N
        
        for k in range(K):
            r_k = resp[:, k]  # (N,)
            
            # --- Update Gaussian parameters ---
            # Mean
            means[k] = np.sum(r_k[:, np.newaxis] * X_cont, axis=0) / N_k[k]
            
            # Covariance
            diff = X_cont - means[k]  # (N, D)
            covs[k] = (diff.T @ (diff * r_k[:, np.newaxis])) / N_k[k]
            # Add regularization
            covs[k] += reg * np.eye(D)
            
            # --- Update categorical parameters ---
            for f_idx in range(F):
                V_f = n_categories[f_idx]
                for v in range(V_f):
                    mask_v = (cat_features[f_idx] == v)  # (N,) bool
                    cat_probs[f_idx][k, v] = (np.sum(r_k[mask_v]) + alpha) / (N_k[k] + alpha * V_f)
    
    # Final assignments
    assignments = np.argmax(resp, axis=1)
    
    print("-" * 60)
    print(f"Final log-likelihood: {log_likelihoods[-1]:.2f}")
    print(f"Cluster sizes: {[np.sum(assignments == k) for k in range(K)]}")
    print(f"Mixing weights: {np.round(pi, 3)}")
    
    return {
        'means': means,
        'covs': covs,
        'cat_probs': cat_probs,
        'pi': pi,
        'resp': resp,
        'assignments': assignments,
        'log_likelihoods': log_likelihoods
    }


# Run EM
cat_features_list = [climate, terrain]
n_cat_list = [N_CLIMATES, N_TERRAINS]

result = mixed_em(X_cont, cat_features_list, n_cat_list, K=5, max_iter=100, seed=123)

## 5. Visualizations

### 5a. Geographic Map with Cluster Assignments

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

resp = result['resp']
assignments = result['assignments']
means = result['means']

# Background shading: posterior probability on a grid
x_range = X_cont[:, 1].min() - 0.3, X_cont[:, 1].max() + 0.3
y_range = X_cont[:, 0].min() - 0.3, X_cont[:, 0].max() + 0.3
gx = np.linspace(x_range[0], x_range[1], 150)
gy = np.linspace(y_range[0], y_range[1], 150)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GY.ravel(), GX.ravel()])  # (lat, lon)

# Compute Gaussian-only posterior on grid (no categorical info for grid)
log_resp_grid = np.zeros((len(grid_pts), K))
for k in range(K):
    log_resp_grid[:, k] = np.log(result['pi'][k] + 1e-300) + \
                           log_gaussian_pdf(grid_pts, means[k], result['covs'][k])
log_max_g = np.max(log_resp_grid, axis=1, keepdims=True)
resp_grid = np.exp(log_resp_grid - log_max_g)
resp_grid /= resp_grid.sum(axis=1, keepdims=True)
grid_assignments = np.argmax(resp_grid, axis=1)

# Color the background
rgba_grid = np.zeros((len(grid_pts), 4))
for k in range(K):
    c = mcolors.to_rgba(COLORS[k])
    mask = grid_assignments == k
    confidence = resp_grid[mask, k]
    for ch in range(3):
        rgba_grid[mask, ch] = c[ch]
    rgba_grid[mask, 3] = 0.08 + 0.15 * confidence

rgba_img = rgba_grid.reshape(150, 150, 4)
ax.imshow(rgba_img, extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
          origin='lower', aspect='auto', interpolation='bilinear')

# Scatter points with climate-based markers
for k in range(K):
    mask_k = assignments == k
    for c_idx in range(N_CLIMATES):
        mask_both = mask_k & (climate == c_idx)
        if np.sum(mask_both) > 0:
            ax.scatter(X_cont[mask_both, 1], X_cont[mask_both, 0],
                       c=COLORS[k], marker=CLIMATE_MARKERS[c_idx],
                       s=50, alpha=0.7, edgecolors='white', linewidths=0.5)

# Learned means as black stars
for k in range(K):
    ax.plot(means[k, 1], means[k, 0], 'k*', markersize=20, markeredgewidth=2,
            markeredgecolor='white', zorder=10)

# Legend for clusters
cluster_handles = [mpatches.Patch(color=COLORS[k], label=f'Component {k}') for k in range(K)]
# Legend for markers (climate)
marker_handles = [Line2D([0], [0], marker=CLIMATE_MARKERS[c], color='gray', linestyle='None',
                          markersize=8, label=CLIMATES[c]) for c in range(N_CLIMATES)]
ax.legend(handles=cluster_handles + marker_handles, loc='upper left', fontsize=9,
          title='Components & Climate', title_fontsize=10)

ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title('Mixed-Type Mixture: Geographic Assignments\n(markers = climate type, background = Gaussian posterior)',
             fontsize=13)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("Stars = learned component means. Marker shapes indicate climate type.")
print("Notice how geographically overlapping clusters are separated by their categorical profiles!")

### 5b. Categorical Distribution Heatmaps

One of the biggest advantages of the mixture model: we can inspect the **learned categorical probabilities** for each component. These tell us the "profile" of each cluster.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

cat_names = ['Climate', 'Terrain']
cat_labels = [CLIMATES, TERRAINS]
cat_probs = result['cat_probs']

# Match learned components to true clusters for comparison
# (find best permutation by Rand Index - approximate via assignment overlap)
from itertools import permutations

def find_best_permutation(true_labels, pred_labels, K):
    """Find the permutation of pred labels that best matches true labels."""
    best_perm = None
    best_match = -1
    for perm in permutations(range(K)):
        remapped = np.zeros_like(pred_labels)
        for k in range(K):
            remapped[pred_labels == k] = perm[k]
        match = np.sum(remapped == true_labels)
        if match > best_match:
            best_match = match
            best_perm = perm
    return best_perm, best_match / len(true_labels)

perm, accuracy = find_best_permutation(true_labels, assignments, K)
print(f"Best label permutation: {perm} (accuracy: {accuracy:.1%})")

for f_idx, ax in enumerate(axes):
    learned = cat_probs[f_idx]  # (K, V_f)
    
    # Reorder rows by best permutation for comparison
    display_order = list(perm)
    
    im = ax.imshow(learned[list(np.argsort(perm))], cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
    
    # Annotate cells
    sorted_indices = list(np.argsort(perm))
    for i, ki in enumerate(sorted_indices):
        for j in range(learned.shape[1]):
            val = learned[ki, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=11,
                    fontweight='bold', color=color)
    
    ax.set_xticks(range(len(cat_labels[f_idx])))
    ax.set_xticklabels(cat_labels[f_idx], fontsize=11)
    ax.set_yticks(range(K))
    ax.set_yticklabels([f'Comp {ki} (true {i})' for i, ki in enumerate(sorted_indices)], fontsize=10)
    ax.set_title(f'Learned {cat_names[f_idx]} Probabilities', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Probability')

plt.suptitle('Learned Categorical Distributions per Component', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print true vs learned comparison
print("\n--- True vs Learned Category Profiles ---")
sorted_indices = list(np.argsort(perm))
for true_k in range(K):
    learned_k = sorted_indices[true_k]
    print(f"\nTrue cluster {true_k} <-> Learned component {learned_k}:")
    print(f"  Climate (true):   {np.round(true_params[true_k]['climate_probs'], 2)}")
    print(f"  Climate (learned):{np.round(cat_probs[0][learned_k], 2)}")
    print(f"  Terrain (true):   {np.round(true_params[true_k]['terrain_probs'], 2)}")
    print(f"  Terrain (learned):{np.round(cat_probs[1][learned_k], 2)}")

### 5c. Soft Assignment Visualization

The mixture model's key advantage: points near cluster boundaries get **soft** assignments reflecting genuine uncertainty. Points that are geographically ambiguous but categorically clear should still have high confidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: opacity = confidence (max responsibility)
ax = axes[0]
confidence = np.max(resp, axis=1)  # (N,)

for k in range(K):
    mask_k = assignments == k
    colors_k = np.array([mcolors.to_rgba(COLORS[k])] * np.sum(mask_k))
    colors_k[:, 3] = confidence[mask_k]  # alpha = confidence
    ax.scatter(X_cont[mask_k, 1], X_cont[mask_k, 0], c=colors_k,
               s=50, edgecolors='gray', linewidths=0.3)

# Learned means
for k in range(K):
    ax.plot(means[k, 1], means[k, 0], 'k*', markersize=18, markeredgewidth=2,
            markeredgecolor='white', zorder=10)

ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title('Soft Assignments (opacity = confidence)', fontsize=13)
ax.grid(True, alpha=0.2)

# Right: highlight uncertain points (boundary region between clusters 1 & 2)
ax = axes[1]
uncertain_mask = confidence < 0.8
certain_mask = ~uncertain_mask

# Plot certain points faintly
for k in range(K):
    mask_k = (assignments == k) & certain_mask
    if np.sum(mask_k) > 0:
        ax.scatter(X_cont[mask_k, 1], X_cont[mask_k, 0], c=COLORS[k],
                   s=30, alpha=0.2, edgecolors='none')

# Plot uncertain points prominently with pie-chart-like coloring
for i in np.where(uncertain_mask)[0]:
    # Color = blend of top-2 component colors weighted by responsibility
    top2 = np.argsort(resp[i])[-2:]
    w1, w2 = resp[i, top2[1]], resp[i, top2[0]]
    c1 = np.array(mcolors.to_rgb(COLORS[top2[1]]))
    c2 = np.array(mcolors.to_rgb(COLORS[top2[0]]))
    blended = w1 * c1 + w2 * c2
    ax.scatter(X_cont[i, 1], X_cont[i, 0], c=[blended], s=80,
               edgecolors='black', linewidths=1.0, zorder=5)

# Mark the overlap zone
c1_center = means[np.argsort(perm)[1]]
c2_center = means[np.argsort(perm)[2]]
mid = 0.5 * (c1_center + c2_center)
ax.annotate('Overlap zone:\npolar vs tropical', xy=(mid[1], mid[0]),
            fontsize=10, ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title(f'Uncertain Points (confidence < 0.8): {np.sum(uncertain_mask)} points', fontsize=13)
ax.grid(True, alpha=0.2)

plt.suptitle('Soft Assignment Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean confidence: {confidence.mean():.3f}")
print(f"Points with confidence > 0.9: {np.sum(confidence > 0.9)} ({np.sum(confidence > 0.9)/N:.1%})")
print(f"Points with confidence < 0.8: {np.sum(confidence < 0.8)} ({np.sum(confidence < 0.8)/N:.1%})")

### 5d. Convergence Plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5))

ll = result['log_likelihoods']
ax.plot(range(len(ll)), ll, 'o-', color='#e74c3c', markersize=4, linewidth=1.5)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Log-Likelihood', fontsize=12)
ax.set_title('EM Convergence: Mixed-Type Mixture Model', fontsize=14)
ax.grid(True, alpha=0.3)

# Annotate start and end
ax.annotate(f'Start: {ll[0]:.0f}', xy=(0, ll[0]), xytext=(5, ll[0] - (ll[-1]-ll[0])*0.1),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'End: {ll[-1]:.1f}', xy=(len(ll)-1, ll[-1]),
            xytext=(len(ll)-1 - len(ll)*0.2, ll[-1] - (ll[-1]-ll[0])*0.1),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

print(f"Converged in {len(ll)} iterations")
print(f"Log-likelihood improved from {ll[0]:.1f} to {ll[-1]:.1f}")

## 6. K-Medoids (Gower) vs Mixed-Type Mixture

Now let's run K-Medoids with Gower distance on the same data and compare. This mirrors the approach from `gower_kmedoids.ipynb`, but here we'll see the mixture model's advantages on data with overlapping geographic clusters.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Haversine distance on unit sphere (inputs in radians)."""
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def gower_distance_matrix(lats, lons, climate, terrain, w_geo=1.0, w_climate=1.0, w_terrain=1.0):
    """
    Compute pairwise Gower distance matrix.
    
    Geographic: haversine / pi (normalized to [0,1])
    Climate: simple matching (0 same, 1 different)
    Terrain: simple matching (0 same, 1 different)
    Gower = weighted average
    """
    N = len(lats)
    D = np.zeros((N, N))
    w_total = w_geo + w_climate + w_terrain
    
    for i in range(N):
        for j in range(i + 1, N):
            # Geographic distance (normalized)
            d_geo = haversine_distance(lats[i], lons[i], lats[j], lons[j]) / np.pi
            
            # Categorical distances (simple matching)
            d_climate = 0.0 if climate[i] == climate[j] else 1.0
            d_terrain = 0.0 if terrain[i] == terrain[j] else 1.0
            
            # Weighted average
            D[i, j] = (w_geo * d_geo + w_climate * d_climate + w_terrain * d_terrain) / w_total
            D[j, i] = D[i, j]
    
    return D


def kmedoids_pam(D, k=5, max_iter=100, seed=123):
    """
    K-Medoids using simplified PAM algorithm.
    
    Parameters:
        D: (N, N) precomputed distance matrix
        k: number of clusters
        max_iter: maximum iterations
        seed: random seed
    
    Returns:
        medoids: indices of medoid points
        labels: cluster assignments
        costs: list of total costs per iteration
    """
    rng = np.random.default_rng(seed)
    N = D.shape[0]
    
    # Initialize: pick k random medoids
    medoids = rng.choice(N, size=k, replace=False)
    
    costs = []
    
    for iteration in range(max_iter):
        # Assign each point to nearest medoid
        dists_to_medoids = D[:, medoids]  # (N, k)
        labels = np.argmin(dists_to_medoids, axis=1)
        
        # Total cost
        cost = sum(D[i, medoids[labels[i]]] for i in range(N))
        costs.append(cost)
        
        # Try swaps: for each medoid, try replacing with each non-medoid
        improved = False
        for m_idx in range(k):
            non_medoids = [i for i in range(N) if i not in medoids]
            # Try a random subset of swaps (simplified PAM)
            candidates = rng.choice(non_medoids, size=min(30, len(non_medoids)), replace=False)
            for candidate in candidates:
                new_medoids = medoids.copy()
                new_medoids[m_idx] = candidate
                
                # Compute new cost
                new_dists = D[:, new_medoids]
                new_labels = np.argmin(new_dists, axis=1)
                new_cost = sum(D[i, new_medoids[new_labels[i]]] for i in range(N))
                
                if new_cost < cost:
                    medoids = new_medoids
                    labels = new_labels
                    cost = new_cost
                    improved = True
        
        if not improved:
            break
    
    return medoids, labels, costs


# Compute Gower distance matrix
print("Computing Gower distance matrix...")
D_gower = gower_distance_matrix(X_cont[:, 0], X_cont[:, 1], climate, terrain)
print(f"Distance matrix shape: {D_gower.shape}")
print(f"Distance range: [{D_gower[D_gower > 0].min():.4f}, {D_gower.max():.4f}]")

# Run K-Medoids
print("\nRunning K-Medoids (PAM)...")
medoids, kmed_labels, kmed_costs = kmedoids_pam(D_gower, k=5, max_iter=50, seed=123)
print(f"K-Medoids converged after {len(kmed_costs)} iterations")
print(f"Final cost: {kmed_costs[-1]:.4f}")
print(f"Cluster sizes: {[np.sum(kmed_labels == k) for k in range(K)]}")

In [ ]:
# Compute Rand Index for comparison
def rand_index(labels_true, labels_pred):
    """
    Compute the Rand Index between two clusterings.
    RI = (number of agreeing pairs) / (total pairs)
    """
    N = len(labels_true)
    a = 0  # pairs in same cluster in both
    b = 0  # pairs in different clusters in both
    total = 0
    
    for i in range(N):
        for j in range(i + 1, N):
            same_true = labels_true[i] == labels_true[j]
            same_pred = labels_pred[i] == labels_pred[j]
            if same_true and same_pred:
                a += 1
            elif not same_true and not same_pred:
                b += 1
            total += 1
    
    return (a + b) / total


ri_mixture = rand_index(true_labels, assignments)
ri_kmedoids = rand_index(true_labels, kmed_labels)

print(f"Rand Index (Mixed-Type Mixture): {ri_mixture:.4f}")
print(f"Rand Index (K-Medoids + Gower):  {ri_kmedoids:.4f}")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: K-Medoids (Gower)
ax = axes[0]
for k in range(K):
    mask = kmed_labels == k
    for c_idx in range(N_CLIMATES):
        mask_both = mask & (climate == c_idx)
        if np.sum(mask_both) > 0:
            ax.scatter(X_cont[mask_both, 1], X_cont[mask_both, 0],
                       c=COLORS[k], marker=CLIMATE_MARKERS[c_idx],
                       s=50, alpha=0.7, edgecolors='white', linewidths=0.5)

# Mark medoids
for k in range(K):
    m = medoids[k]
    ax.plot(X_cont[m, 1], X_cont[m, 0], 'k*', markersize=18,
            markeredgewidth=2, markeredgecolor='white', zorder=10)

ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title(f'K-Medoids (Gower Distance)\nRand Index: {ri_kmedoids:.4f}', fontsize=13)
ax.grid(True, alpha=0.3)

# Right: Mixed-Type Mixture
ax = axes[1]
for k in range(K):
    mask = assignments == k
    for c_idx in range(N_CLIMATES):
        mask_both = mask & (climate == c_idx)
        if np.sum(mask_both) > 0:
            ax.scatter(X_cont[mask_both, 1], X_cont[mask_both, 0],
                       c=COLORS[k], marker=CLIMATE_MARKERS[c_idx],
                       s=50, alpha=0.7, edgecolors='white', linewidths=0.5)

# Learned means
for k in range(K):
    ax.plot(means[k, 1], means[k, 0], 'k*', markersize=18,
            markeredgewidth=2, markeredgecolor='white', zorder=10)

ax.set_xlabel('Longitude (rad)', fontsize=12)
ax.set_ylabel('Latitude (rad)', fontsize=12)
ax.set_title(f'Mixed-Type Mixture (EM)\nRand Index: {ri_mixture:.4f}', fontsize=13)
ax.grid(True, alpha=0.3)

# Shared legend
marker_handles = [Line2D([0], [0], marker=CLIMATE_MARKERS[c], color='gray', linestyle='None',
                          markersize=8, label=CLIMATES[c]) for c in range(N_CLIMATES)]
axes[1].legend(handles=marker_handles, loc='upper left', fontsize=10, title='Climate Type')

plt.suptitle('K-Medoids (Gower) vs Mixed-Type Mixture (EM)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey observation: In the overlapping region (lon~0.5, lat~0.75),")
print("the mixture model separates polar/mountain from tropical/forest stations")
print("by leveraging the categorical distributions, not just geographic proximity.")

## Choosing K — PVE & Silhouette Analysis

How do we know K = 5 is the right number of clusters? We use two metrics:

**PVE (Proportion of Variance Explained)**

$$\text{PVE}(K) = 1 - \frac{\text{TWCSS}_K}{\text{TWCSS}_1}$$

where $\text{TWCSS}_K$ is the **total within-cluster sum of squared distances** for K clusters and $\text{TWCSS}_1$ is the single-cluster baseline (all points assigned to one cluster). PVE ranges from 0 to 1; an "elbow" in the curve marks a natural K choice.

**Silhouette Score**

For each point $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, +1]$$

where $a(i)$ = mean distance to other points in the **same** cluster (cohesion), and $b(i)$ = mean distance to points in the **nearest other** cluster (separation). Computed from the precomputed pairwise distance matrix so it automatically respects this notebook's metric.

In [ ]:
def silhouette_from_matrix(D, labels):
    """
    Mean silhouette score from a precomputed N×N distance matrix.
    s(i) = (b(i) - a(i)) / max(a(i), b(i))
    """
    n = len(labels)
    unique_clusters = np.unique(labels)
    if len(unique_clusters) <= 1:
        return 0.0
    scores = np.zeros(n)
    for i in range(n):
        ci = labels[i]
        same_mask = (labels == ci).copy()
        same_mask[i] = False
        a_i = float(np.mean(D[i, same_mask])) if np.any(same_mask) else 0.0
        b_i = np.inf
        for cj in unique_clusters:
            if cj == ci:
                continue
            other_mask = (labels == cj)
            if np.any(other_mask):
                b_i = min(b_i, float(np.mean(D[i, other_mask])))
        denom = max(a_i, b_i)
        scores[i] = (b_i - a_i) / denom if denom > 1e-15 else 0.0
    return float(np.mean(scores))

In [ ]:
# For the mixed-type mixture, use hard assignments (result['assignments']) and
# Gower distance for TWCSS and silhouette.
# gower_distance_matrix() and mixed_em() are already defined above.

# Build full N×N Gower distance matrix for silhouette
print("Building pairwise Gower distance matrix (this may take a moment)...")
lats_cont = X_cont[:, 0]
lons_cont = X_cont[:, 1]
D_gower_full = gower_distance_matrix(lats_cont, lons_cont, climate, terrain)

# TWCSS baseline: single global medoid (point minimising total Gower distance)
global_med_idx = int(np.argmin(D_gower_full.sum(axis=1)))
TWCSS_1 = float(np.sum(D_gower_full[:, global_med_idx] ** 2))

K_range = range(2, 11)
pve_vals = []
sil_vals = []

print(f"{'K':>4} {'TWCSS':>10} {'PVE':>8} {'Silhouette':>12}")
print("-" * 38)
for k in K_range:
    res_k = mixed_em(X_cont, cat_features_list, n_cat_list, K=k, max_iter=50, seed=123)
    lbls_k = res_k['assignments']
    means_k = res_k['means']
    # TWCSS: Gower distances from each point to its cluster medoid
    # (find medoid per cluster, then sum squared distances)
    twcss_k = 0.0
    for c in range(k):
        mask_c = (lbls_k == c)
        if not np.any(mask_c):
            continue
        idx_c = np.where(mask_c)[0]
        # Medoid of cluster c: minimises sum of intra-cluster Gower distances
        sub_D = D_gower_full[np.ix_(idx_c, idx_c)]
        local_med = idx_c[int(np.argmin(sub_D.sum(axis=1)))]
        twcss_k += float(np.sum(D_gower_full[idx_c, local_med] ** 2))
    pve = 1 - twcss_k / TWCSS_1
    sil = silhouette_from_matrix(D_gower_full, lbls_k)
    pve_vals.append(pve)
    sil_vals.append(sil)
    print(f"{k:>4} {twcss_k:>10.4f} {pve:>8.4f} {sil:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
K_chosen = 5
K_vals = list(K_range)

ax = axes[0]
ax.plot(K_vals, pve_vals, 'o-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#3498db', markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(K_chosen, color='#e74c3c', linestyle='--', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, pve_vals, alpha=0.08, color='#3498db')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('PVE', fontsize=12)
ax.set_title('PVE Elbow Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(K_vals, sil_vals, 's-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#9b59b6', markeredgecolor='white', markeredgewidth=1.5)
best_k_sil = K_vals[int(np.argmax(sil_vals))]
ax.axvline(best_k_sil, color='#9b59b6', linestyle='--', linewidth=1.8,
           label=f'Best silhouette K = {best_k_sil}')
ax.axvline(K_chosen, color='#e74c3c', linestyle=':', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, sil_vals, alpha=0.08, color='#9b59b6')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

plt.suptitle('Choosing K: PVE & Silhouette Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nK=2..10 summary:")
print(f"{'K':>4} {'PVE':>8} {'Silhouette':>12}")
print("-" * 27)
for k, pve, sil in zip(K_vals, pve_vals, sil_vals):
    marker = " ◄ chosen" if k == K_chosen else ""
    print(f"{k:>4} {pve:>8.4f} {sil:>12.4f}{marker}")

## 7. Summary Table

| Aspect | K-Medoids (Gower) | Mixed-Type Mixture (EM) |
|--------|-------------------|------------------------|
| **Assignment** | Hard (each point to one cluster) | Soft (probability per cluster) |
| **Distance / Likelihood** | Gower distance (heuristic blend) | Joint Gaussian + Multinomial likelihood |
| **Center type** | Medoid (actual data point) | Mean (continuous) + category distributions |
| **Handles category probabilities** | No — only same/different | Yes — learns full distribution per component |
| **Uncertainty** | None | Posterior responsibility per point |
| **Best for** | Quick exploration, no parametric assumptions | Generative modeling, uncertainty quantification |

### Key Takeaways

1. **Mixed-type mixture gives a proper generative model** for heterogeneous data. Each cluster is described by a Gaussian (where) and multinomials (what kind), rather than just a center point.

2. **The categorical distributions reveal which categories are associated with which geographic regions.** We can inspect $\phi_{k,f,v}$ to understand cluster profiles — e.g., "Component 2 is 70% tropical climate and 70% forest terrain."

3. **Soft assignments handle ambiguous points naturally.** A station in the overlap zone between polar/mountain and tropical/forest clusters gets split responsibility reflecting genuine uncertainty in geographic space, resolved by its categorical attributes.

4. **K-Medoids is simpler and assumption-free.** It doesn't require Gaussian-shaped continuous features or conditionally independent categories. When parametric assumptions are questionable, K-Medoids with Gower distance is a safer choice.

---

**Series note:** This notebook extends the mixed-type clustering story from `gower_kmedoids.ipynb` (distance-based) to a full probabilistic model (likelihood-based). Together with `euclidean_gmm.ipynb` (continuous-only EM) and `spherical_vmf_mixture.ipynb` (directional EM), this completes the progression from hard geometric clustering to soft probabilistic models across different data types and geometries.